# Pipeline End-to-End — Node-by-Node Trace

Notebook chay toan bo pipeline EduBot, phan ro **tung node** de kiem soat:

```
1 ContextAnalyzer -> 2 IntentRouter (LLM) -> 3 SessionManager -> 4 ActionPlanner -> 5 RAG Search -> 6 Handler -> 7 Session Save
```

Ket qua pipeline duoc ghi ra `pipeline_trace.log` dang **JSON** (reset moi query). File `app.log` giu persistent log.

---
## 0. Setup — Path & Environment

In [1]:
import sys
import os
import time
import json
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent.parent
print(f"Project root: {PROJECT_ROOT}")

# Add src to path
for p in [str(PROJECT_ROOT), str(PROJECT_ROOT / 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Load env
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

print(f"API Key: {'SET' if os.getenv('GENAI_API_KEY') else 'NOT SET'}")
print(f"Python: {sys.version.split()[0]}")

Project root: c:\Users\Admin\OneDrive - Hanoi University of Science and Technology\Desktop\DATN
API Key: SET
Python: 3.12.4


---
## 1. Init — CustomSearch + Reranker + Orchestrator

In [2]:
from src.config.config import settings
from src.rag.retrieve_rebuild import CustomSearch
from src.rag.reranker import Reranker
from src.llm.orchestrator import Orchestrator

DATA_DIR = PROJECT_ROOT / 'data'
CHUNKS_PATH = str(DATA_DIR / 'rag_chunks_v2.json')
EMBEDDINGS_PATH = str(DATA_DIR / 'embeddings.npy')

print("=" * 60)
print("[NODE 0] Initializing Components")
print("=" * 60)

# 1a. CustomSearch
t0 = time.time()
searcher = CustomSearch(chunks_path=CHUNKS_PATH, embeddings_path=EMBEDDINGS_PATH)
print(f"  CustomSearch: {searcher.corpus_size} chunks, dim={searcher.embeddings.shape[1]} ({time.time()-t0:.2f}s)")

# 1b. Reranker
reranker = Reranker()
print(f"  Reranker: {settings.RERANKER_MODEL} (lazy load)")

# 1c. Orchestrator
orch = Orchestrator(retriever=searcher, reranker=reranker)
print(f"  Orchestrator: ready")
print(f"  LLM Model: {settings.LLM_MODEL}")
print("=" * 60)

[17:46:26] INFO    | Tokenizing 2348 docs with underthesea...


[NODE 0] Initializing Components
CustomSearch initialized: 2348 docs, vocab=9672, avgdl=137.7
  CustomSearch: 2348 chunks, dim=768 (15.37s)
  Reranker: AITeamVN/Vietnamese_Reranker (lazy load)
  Orchestrator: ready
  LLM Model: gemini-2.5-flash-lite


---
## 2. Helper — Log Viewer & Node Runner

In [3]:
TRACE_LOG = PROJECT_ROOT / 'logs' / 'pipeline_trace.log'
APP_LOG = PROJECT_ROOT / 'logs' / 'app.log'

def show_trace_log():
    """Hien thi pipeline_trace.log (JSON format)."""
    if TRACE_LOG.exists():
        content = TRACE_LOG.read_text(encoding='utf-8')
        try:
            data = json.loads(content)
            print(json.dumps(data, indent=2, ensure_ascii=False, default=str))
        except json.JSONDecodeError:
            print(content)  # fallback to raw text
    else:
        print('[!] pipeline_trace.log chua ton tai')

def load_trace_json() -> dict:
    """Load pipeline_trace.log as dict."""
    if TRACE_LOG.exists():
        return json.loads(TRACE_LOG.read_text(encoding='utf-8'))
    return {}

def show_app_log(n=30):
    """Hien thi n dong cuoi cua app.log."""
    if APP_LOG.exists():
        lines = APP_LOG.read_text(encoding='utf-8').strip().split('\n')
        for line in lines[-n:]:
            print(line)
    else:
        print('[!] app.log chua ton tai')

def show_debug_info(debug_info: dict):
    """Hien thi debug info tu orchestrator.last_debug_info."""
    if not debug_info:
        print('[!] Chua co debug info')
        return
    print(json.dumps(debug_info, indent=2, ensure_ascii=False, default=str))

print('Helpers loaded: show_trace_log(), load_trace_json(), show_app_log(n), show_debug_info(info)')

Helpers loaded: show_trace_log(), load_trace_json(), show_app_log(n), show_debug_info(info)


---
## 3. Multi-turn E2E Test

Chạy thử nghiệm hội thoại nhiều lượt (Multi-turn) để xem Orchestrator bắt context và phản hồi thế nào, đồng thời in ra cục JSON Debug Info.

In [4]:
import json, time

# Khởi tạo Orchestrator sạch cho Test
orch = Orchestrator(retriever=searcher, reranker=reranker)

queries = [
    "Xin chào, bạn là ai?",
    "Giải thích cho tôi về mạng LAN",
    "Nó khác WAN như thế nào?"
]

for i, q in enumerate(queries, 1):
    print("=" * 80)
    print(f"\n[TURN {i}] QUERY: {q}")
    print("=" * 80)
    
    # Chạy pipeline
    response_chunks = []
    t0 = time.time()
    for chunk in orch.ask(q, ui_book="KNTT"):
        response_chunks.append(chunk)
    total_time = time.time() - t0
    
    # In kết quả
    print(f"\n--- RESPONSE ({total_time:.2f}s) ---")
    full_res = "".join(response_chunks)
    print(full_res[:500] + ("..." if len(full_res) > 500 else ""))
    
    # In Debug JSON
    print(f"\n--- DEBUG INFO JSON ---")
    show_debug_info(orch.last_debug_info)
    print("\n")


[17:47:01] INFO    | ============================================================
[17:47:01] INFO    | QUERY [6632a072]: 'Xin chào, bạn là ai?'



[TURN 1] QUERY: Xin chào, bạn là ai?


[17:47:01] INFO    | IntentRouter: intent=chat, task_type=None, topic=None, is_new_topic=False, book=None
[17:47:01] INFO    | IntentRouter (0.69s): intent=chat, task_type=None, topic=None
[17:47:01] INFO    | No current session, creating new
[17:47:01] INFO    | New session created: id=91483035, topic='', intent=chat, book=None
[17:47:01] INFO    | Session: id=91483035, topic='', msgs=0
[17:47:01] INFO    | ActionPlan: chat (Default chat intent)
[17:47:01] INFO    | Book: ui=KNTT, llm=None, session=KNTT -> effective=KNTT
[17:47:01] INFO    | RAGAgent: book filter='KNTT' → 1144 chunks in scope
[17:47:01] INFO    | RAGAgent: strategy=standard | grade=None | topic=None | book=KNTT | Query cụ thể, không có grade/topic context


Loading embedding model: dangvantuan/vietnamese-document-embedding...
Model loaded on cuda
Loading reranker: AITeamVN/Vietnamese_Reranker...
Reranker loaded on cuda


[17:47:31] INFO    | RAGAgent done: 5 chunks, 30.00s
[17:47:33] INFO    | Total time: 31.85s
[17:47:33] INFO    | ============================================================
[17:47:33] INFO    | ============================================================
[17:47:33] INFO    | QUERY [450cccca]: 'Giải thích cho tôi về mạng LAN'
[17:47:33] INFO    | ContextAnalyzer: enriched query with history



--- RESPONSE (31.85s) ---
Chào bạn! 👋 Mình là EduBot, trợ lý học tập Tin học THPT Việt Nam. Mình ở đây để giúp bạn hiểu rõ hơn về các kiến thức Tin học theo sách giáo khoa nhé! 😊

--- DEBUG INFO JSON ---
{
  "request_id": "6632a072",
  "query": "Xin chào, bạn là ai?",
  "timestamp": "2026-04-12 17:47:01",
  "effective_book": "KNTT",
  "steps": [
    {
      "node": "ContextAnalyzer",
      "enriched": false,
      "rewrite": null
    },
    {
      "node": "IntentRouter",
      "primary_intent": "chat",
      "task_type": null,
      "topic": null,
      "is_new_topic": false,
      "book": null,
      "time_s": 0.69
    },
    {
      "node": "SessionManager",
      "session_id": "91483035",
      "topic": "",
      "intent": "chat",
      "book": null,
      "total_messages": 0,
      "has_quiz_state": false,
      "has_slide_state": false
    },
    {
      "node": "ActionPlanner",
      "action": "chat",
      "reason": "Default chat intent",
      "round_id": null
    },
    {
  

[17:47:33] INFO    | QueryRewriter: needs_rewrite=False, queries=['Giải thích mạng LAN']
[17:47:33] INFO    | QueryRewriter (0.90s): 1 queries → ['Giải thích mạng LAN']
[17:47:34] INFO    | IntentRouter: intent=explain, task_type=None, topic=Mạng LAN, is_new_topic=True, book=None
[17:47:34] INFO    | IntentRouter (0.80s): intent=explain, task_type=None, topic=Mạng LAN
[17:47:34] INFO    | Topic changed: '' -> 'Mạng LAN', creating new session
[17:47:34] INFO    | New session created: id=34000fb2, topic='Mạng LAN', intent=explain, book=None
[17:47:34] INFO    | Session: id=34000fb2, topic='Mạng LAN', msgs=0
[17:47:34] INFO    | ActionPlan: explain_concept (General concept explanation)
[17:47:34] INFO    | Book: ui=KNTT, llm=None, session=KNTT -> effective=KNTT
[17:47:34] INFO    | RAGAgent: book filter='KNTT' → 1144 chunks in scope
[17:47:34] INFO    | RAGAgent: strategy=broad | grade=None | topic=Mạng LAN | book=KNTT | Query tổng quát: broad=False, grade_only=False, topic_broad=True
[17


--- RESPONSE (7.56s) ---
Đang tìm tài liệu để giải thích...

Chào bạn! Mình là EduBot, rất vui được giải thích về mạng LAN cho bạn đây. 😊

### 1. Khái niệm cốt lõi

**Mạng LAN (Local Area Network)** là một mạng máy tính kết nối các thiết bị trong một khu vực địa lý nhỏ, chẳng hạn như một văn phòng, một tòa nhà, hoặc một khuôn viên trường học. Mục đích chính của mạng LAN là cho phép các thiết bị này chia sẻ tài nguyên và giao tiếp với nhau một cách hiệu quả.

### 2. Giải thích chi tiết

Để hiểu rõ hơn về mạng LAN, chúng ...

--- DEBUG INFO JSON ---
{
  "request_id": "450cccca",
  "query": "Giải thích cho tôi về mạng LAN",
  "timestamp": "2026-04-12 17:47:33",
  "effective_book": "KNTT",
  "steps": [
    {
      "node": "ContextAnalyzer",
      "enriched": true,
      "rewrite": {
        "rewritten_queries": [
          "Giải thích mạng LAN"
        ],
        "time_s": 0.9
      }
    },
    {
      "node": "IntentRouter",
      "primary_intent": "explain",
      "task_type": null,
  

[17:47:41] INFO    | QueryRewriter: needs_rewrite=True, queries=['Sự khác nhau giữa mạng LAN và mạng WAN', 'So sánh đặc điểm của mạng LAN và mạng WAN', 'Phân biệt mạng cục bộ (LAN) và mạng diện rộng (WAN)']
[17:47:41] INFO    | QueryRewriter (1.05s): 3 queries → ['Sự khác nhau giữa mạng LAN và mạng WAN', 'So sánh đặc điểm của mạng LAN và mạng WAN', 'Phân biệt mạng cục bộ (LAN) và mạng diện rộng (WAN)']
[17:47:42] INFO    | IntentRouter: intent=explain, task_type=None, topic=So sánh LAN và WAN, is_new_topic=False, book=None
[17:47:42] INFO    | IntentRouter (1.01s): intent=explain, task_type=None, topic=So sánh LAN và WAN
[17:47:42] INFO    | Session: id=34000fb2, topic='Mạng LAN', msgs=2
[17:47:42] INFO    | ActionPlan: explain_concept (General concept explanation)
[17:47:42] INFO    | Book: ui=KNTT, llm=None, session=KNTT -> effective=KNTT
[17:47:42] INFO    | RAGAgent: book filter='KNTT' → 1144 chunks in scope
[17:47:42] INFO    | RAGAgent: strategy=broad | grade=None | topic=So sánh


--- RESPONSE (12.95s) ---
Đang tìm tài liệu để giải thích...

Chào em, rất vui được giải thích cho em về sự khác biệt giữa mạng LAN và WAN nhé! Em đã hiểu về mạng LAN rồi, giờ mình cùng tìm hiểu về WAN và xem nó khác gì với LAN nhé.

### 1. Khái niệm cốt lõi

*   **Mạng LAN (Local Area Network)**: Là mạng kết nối các thiết bị trong một phạm vi địa lý **hẹp**, ví dụ như trong một văn phòng, một tòa nhà, hoặc một trường học.
*   **Mạng WAN (Wide Area Network)**: Là mạng kết nối các thiết bị trên một phạm vi địa lý **rộng l...

--- DEBUG INFO JSON ---
{
  "request_id": "f728bd98",
  "query": "Nó khác WAN như thế nào?",
  "timestamp": "2026-04-12 17:47:40",
  "effective_book": "KNTT",
  "steps": [
    {
      "node": "ContextAnalyzer",
      "enriched": true,
      "rewrite": {
        "rewritten_queries": [
          "Sự khác nhau giữa mạng LAN và mạng WAN",
          "So sánh đặc điểm của mạng LAN và mạng WAN",
          "Phân biệt mạng cục bộ (LAN) và mạng diện rộng (WAN)"
        ],
 